# Five tasks without the task instruction: the two published methods, 100 epochs

**This notebook repeats the earlier no-instruction run of InfLoRA and GainLoRA (official code, seed 42, t5-base) with the PUBLISHED
number of epochs: 100 per task instead of 10.** Everything else is as in the 10-epoch run: the task definition is removed from every
input (the "Definition" of every data file of the five tasks becomes the empty string before anything is trained), the same
five tasks, the same learning rate, batch size (effective 32), maximum lengths and thresholds. The official script ties the
development-set check and the checkpoint to the number of epochs (one check and one saved checkpoint every 5 epochs, the best
one by development ROUGE-L is kept), so with 100 epochs the best of 20 checkpoints is kept per task, with 10 epochs the best of 2.

**Nothing has to be edited.** On Kaggle set *Accelerator* to **GPU T4 x2** and *Internet* to **on**, then **Save Version -> Save & Run All**.
The two methods run at the same time, one on each GPU. The 10-epoch run took 36 minutes per method; with ten times the training this
takes about **4.5 to 5 hours** (training scales with the epochs; set-up, test and re-test do not). Nothing in the notebook has a time
limit of its own. After every finished task the task's results are copied to the output folder and the small-files zip
`/kaggle/working/silent_adapters/superni_noinstruction5_published_t5base_e100_small_files.zip` is refreshed, so an interrupted run keeps its finished tasks.
After each task every test input is answered again (original wording only) and the answers that changed since the task was learned
are counted, as before. The printed summary at the end shows these runs next to the 10-epoch no-definition numbers of the same seed.

In [ ]:
import os, re, sys, json, time, shlex, shutil, subprocess

PRESET = "five"       # "five": the five short tasks.  "fifteen": the whole SuperNI order 1 (15 tasks), `certified` only.

CFG = dict(
    run_tag="superni_noinstruction5_published_t5base_e100",   # folder name on Drive. Change it to start a new, separate run.
    model="t5-base",
    epochs=100,
    seeds=[42],                            # one full run per seed; for more seeds write for example [43, 44]
    tasks=["task1510_evalution_relation_extraction", "task363_sst2_polarity_classification",
           "task748_glucose_reverse_cause_event_detection", "task1687_sentiment140_classification",
           "task875_emotion_classification"],
    methods=["inflora", "gainlora_inflora"],   # the two published methods, 100 epochs, one per GPU
    only_if_finished=[],                   # nothing is only shown here
    # here: the re-test of `certified2` already contains plain adapters (`all`) and the exact switch (`key`)
    train_batch=16,                        # halved automatically if the GPU runs out of memory
    eval_batch=16,
    silent=dict(SILENT_LAMBDA="1.0", SILENT_POOL="400", SILENT_NEG_BATCH="8", SILENT_MAX_LEN="256", SILENT_KINDS="a,b"),
    silent2=dict(SILENT_LAMBDA="1.0", SILENT_POOL="800", SILENT_NEG_BATCH="8", SILENT_MAX_LEN="256", SILENT_KINDS="a,b,c,c"),
    locked=dict(SILENT_LAMBDA="0", SILENT_LOCK="1"),
    certified2=dict(SILENT_LAMBDA="0", SILENT_CLAIM="1", SILENT_KAPPA="0.5", SILENT_SIF="0.001", SILENT_EVAL="1"),
    certified=None,                        # first version (code of commit 7892ef6): only shown if finished, never trained again
    retest_baselines=True,                 # also test the published methods with reworded task definitions
    inflora=dict(SILENT_BASELINE="1"), gainlora_inflora=dict(SILENT_BASELINE="1"),
    keep_all_files=False,
    use_all_gpus=True,                     # with two GPUs (Kaggle "T4 x2") two methods are trained at the same time
    drive_dir="/content/drive/MyDrive/silent_adapters",
    work_dir="/content/silent_adapters_work",
    repo_url="https://github.com/liangyanshuo/gainlora",
    repo_commit="e9d0d700538c8d12892f7fd52b216291f7e42acc",   # the exact version of the official code
)

# --- used only for the CPU self-test of this notebook; has no effect in Colab ---
TEST = os.environ.get("S0_TEST") == "1"
if TEST and os.environ.get("S0_PRESET"):
    PRESET = os.environ["S0_PRESET"]

if PRESET == "fifteen":                    # the 15 tasks of SuperNI order 1, in the official order
    CFG.update(run_tag="superni_o1_t5base_e10_ours", methods=["certified2", "inflora", "gainlora_inflora"], tasks=[
        "task1572_samsum_summary", "task363_sst2_polarity_classification", "task1290_xsum_summarization",
        "task181_outcome_extraction", "task002_quoref_answer_generation", "task1510_evalution_relation_extraction",
        "task639_multi_woz_user_utterance_generation", "task1729_personachat_generate_next",
        "task073_commonsenseqa_answer_generation", "task1590_diplomacy_text_generation",
        "task748_glucose_reverse_cause_event_detection", "task511_reddit_tifu_long_text_summarization",
        "task591_sciq_answer_generation", "task1687_sentiment140_classification", "task875_emotion_classification"])

if TEST:
    CFG.update(run_tag="superni_quick_selftest" + ("15" if PRESET == "fifteen" else ""), model=os.environ["S0_MODEL"],
               epochs=1, tasks=CFG["tasks"][:3],
               drive_dir=os.environ["S0_DRIVE"], work_dir=os.environ["S0_WORK"])
    CFG["silent"].update(SILENT_POOL="16", SILENT_NEG_BATCH="4", SILENT_SELFCHECK="9")
    CFG["silent2"].update(SILENT_POOL="16", SILENT_NEG_BATCH="4")
    CFG["certified2"].update(SILENT_BG="1200")
    CFG["only_if_finished"] = ["certified"]
    CFG["train_batch"] = 8
    if os.environ.get("S0_METHODS"):
        CFG["methods"] = os.environ["S0_METHODS"].split(",")
    if os.environ.get("S0_SEEDS"):
        CFG["seeds"] = [int(x) for x in os.environ["S0_SEEDS"].split(",")]

IN_COLAB = "google.colab" in sys.modules
ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or os.path.isdir("/kaggle/working")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
elif ON_KAGGLE and not TEST:                # Kaggle: results go to the notebook's output folder, the rest to a temporary one
    CFG.update(drive_dir="/kaggle/working/silent_adapters",
               work_dir=("/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp") + "/silent_adapters_work")
SAVE_DIR = os.path.join(CFG["drive_dir"], CFG["run_tag"])
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(CFG["work_dir"], exist_ok=True)
print("results are saved in:", SAVE_DIR)

## Set-up
The official code needs old library versions (Python 3.10, PyTorch 2.1.0, Transformers 4.30.2).
This cell builds a separate Python environment with exactly those versions, so Colab's own
libraries are left alone. It takes a few minutes and is repeated after every disconnect.

In [ ]:
def sh(cmd):
    """Run a shell command. Its output is shown only if it fails."""
    print("$", cmd if len(cmd) < 160 else cmd[:157] + "...", flush=True)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print((r.stdout + r.stderr)[-4000:])
        raise RuntimeError("this command failed (its output is printed above): " + cmd[:120])


def env_python(code):
    """Run a short piece of Python in the separate environment. Returns (worked, what it printed)."""
    r = subprocess.run([ENV_PY, "-c", code], capture_output=True, text=True, env=RUN_ENV)
    return r.returncode == 0, (r.stdout + r.stderr).strip()


if not TEST:                                          # check the GPU first, before spending minutes on set-up
    gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True,
                         capture_output=True, text=True).stdout.strip()
    assert gpu, ("No GPU. On Colab: Runtime -> Change runtime type -> T4 GPU. On Kaggle: panel on the right -> "
                 "Session options -> Accelerator -> GPU T4 x2. Then run all again.")
    print("GPU:", gpu.replace("\n", " | "))
N_GPUS = int(os.environ.get("S0_GPUS") or (1 if TEST else len(gpu.splitlines())))

REPO = os.environ.get("S0_REPO") or os.path.join(CFG["work_dir"], "gainlora")
ENV_PY = os.environ.get("S0_ENV_PY") or os.path.join(CFG["work_dir"], "env", "bin", "python")
ENV_DIR = os.path.dirname(os.path.dirname(ENV_PY))

if not os.path.isdir(os.path.join(REPO, "src")):
    shutil.rmtree(REPO, ignore_errors=True)               # anything left from a download that failed half-way
    try:
        sh(f"git clone {CFG['repo_url']} {REPO}")
    except RuntimeError:
        import socket
        try:
            socket.create_connection(("github.com", 443), timeout=10).close()
        except OSError:
            raise RuntimeError(
                "NO INTERNET: the notebook cannot download the code. On Kaggle the internet is off until you switch it on: "
                "panel on the right -> Session options (or Notebook options) -> Internet -> on. The switch appears only "
                "after your phone number is verified (kaggle.com -> Settings -> Phone verification). Then run all again. "
                "Google Drive is not needed on Kaggle.") from None
        raise
    sh(f"git -C {REPO} checkout {CFG['repo_commit']}")
head = subprocess.run(f"git -C {REPO} rev-parse HEAD", shell=True, capture_output=True, text=True).stdout.strip()
assert head == CFG["repo_commit"], f"unexpected version of the official code: {head}"

if not os.path.exists(ENV_PY):
    sh(f"{sys.executable} -m pip install -q uv")
    sh(f"uv venv --python 3.10.12 {ENV_DIR}")
    sh(f"uv pip install -q --python {ENV_PY} torch==2.1.0 --index-url https://download.pytorch.org/whl/cu121")
    sh(f"uv pip install -q --python {ENV_PY} transformers==4.30.2 datasets==2.14.6 accelerate==0.24.1 numpy==1.22.4 "
       "fsspec==2023.10.0 pyarrow==14.0.2 loralib nltk ipdb sentencepiece tabulate absl-py six rouge-score 'protobuf<4'")
    try:                                              # CuPy is optional: see below
        sh(f"uv pip install -q --python {ENV_PY} cupy-cuda12x==12.1.0 numpy==1.22.4")
    except RuntimeError:
        print("CuPy could not be installed; PyTorch will be used for that step instead.")

# The GPU libraries that came with PyTorch, so that CuPy can find them whatever Colab has installed.
import glob
nvidia_libs = sorted(glob.glob(os.path.join(ENV_DIR, "lib", "python3.10", "site-packages", "nvidia", "*", "lib")))
RUN_ENV = dict(os.environ, TOKENIZERS_PARALLELISM="false", WANDB_DISABLED="true", CUDA_VISIBLE_DEVICES="0",
               HF_HOME=os.path.join(CFG["work_dir"], "hf_cache"),
               LD_LIBRARY_PATH=":".join(nvidia_libs + [os.environ.get("LD_LIBRARY_PATH", "")]).strip(":"))

ok, out = env_python("import torch, transformers, datasets, numpy; "
                     "print('torch', torch.__version__, '| transformers', transformers.__version__, '| datasets', "
                     "datasets.__version__, '| numpy', numpy.__version__, '| GPU seen by PyTorch:', "
                     "torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')")
print(out)
assert ok, "The Python environment is broken; the error is printed above."
assert TEST or "NONE" not in out, "PyTorch in the separate environment cannot see the GPU; the details are printed above."

# The official code uses the CuPy library for one step (finding the main directions of a matrix, an
# SVD, on the GPU). If CuPy does not work on this machine, the same step is done with PyTorch.
FALLBACK = os.path.join(CFG["work_dir"], "cupy_fallback")
FALLBACK_CODE = """\
# Stand-in for CuPy that does the same maths with PyTorch. Arrays are ordinary PyTorch tensors.
import torch
from torch.utils.dlpack import to_dlpack as _to, from_dlpack as _from
torch.Tensor.toDlpack = lambda self: _to(self)
def fromDlpack(capsule): return _from(capsule)
def dot(a, b): return a @ b
def hstack(parts): return torch.hstack(tuple(parts))
def ones(shape): return torch.ones(shape, device="cuda" if torch.cuda.is_available() else "cpu")
class linalg:
    @staticmethod
    def svd(a, full_matrices=True): return torch.linalg.svd(a, full_matrices=full_matrices)
"""
cupy_test = "import cupy as cp; u, s, v = cp.linalg.svd(cp.ones((4, 4)), full_matrices=False); print('ok', float((s ** 2).sum()))"
ok, out = (False, "forced for the self-test") if os.environ.get("S0_FORCE_FALLBACK") else env_python(cupy_test)
if ok and "cupy_fallback" not in out:
    SVD_BACKEND = "cupy"
else:
    print("CuPy does not work here. Reason:\n   ", "\n    ".join(out.splitlines()[-6:]))
    os.makedirs(os.path.join(FALLBACK, "cupy"), exist_ok=True)
    open(os.path.join(FALLBACK, "cupy", "__init__.py"), "w").write(FALLBACK_CODE)
    RUN_ENV["PYTHONPATH"] = ":".join([FALLBACK, RUN_ENV.get("PYTHONPATH", "")]).strip(":")
    ok, out = env_python(cupy_test)
    assert ok, "The PyTorch stand-in for CuPy failed too:\n" + out
    SVD_BACKEND = "pytorch (stand-in for cupy)"
print("SVD step done with:", SVD_BACKEND)

## Our method's code
`silent_run.py` starts the official training script unchanged and adds one extra loss: on stand-in
inputs, the model with the new adapter must answer exactly as it did without it. The stand-in inputs
are the task's own inputs under an unrelated task definition, and text from unrelated tasks. No
other task of this benchmark is used, and nothing about old tasks is stored.

In [ ]:
SILENT_RUN_SOURCE = r'''"""Silent adapters: runs the official GainLoRA script `src/run_t5.py` unchanged, with silence training added.

Run it from the folder of the official code, with the same arguments as `src/run_t5.py` and
`--model_name olora --lambda1 0 --lambda2 0` (plain adapters: one new LoRA per task, older ones frozen,
all added together, no gate). This file adds one extra loss while a task is being trained:

    on stand-in inputs, the model WITH the new adapter must give the same next-word probabilities
    as the model WITHOUT it.

Stand-in inputs never use another task of the benchmark:
  kind a: the current task's own inputs, with the task definition replaced by the definition of an
          unrelated task;
  kind b: text of an unrelated task with that task's own definition.
The unrelated tasks come from a different benchmark shipped in the same repository (Long_Sequence),
leaving out its sentiment tasks because the SuperNI sequence contains sentiment tasks.

A third kind of stand-in (kind c) is the task's own input under a "definition" that is a random piece
of unrelated text, so the adapter meets a different wrong definition almost every time.

Separately, SILENT_LOCK=1 adds an exact switch. Every input of this benchmark starts with its task
definition. When a task is learned, the tokens of its definition are stored as a key. An input whose
first tokens equal a stored key is answered with every adapter added AFTER that task switched off, so
the model computes exactly what it computed right after learning that task. Inputs that match no key
are answered with all adapters on.

SILENT_CLAIM=1 is the certified switch, which does not need the exact wording. Each task stores a
signature: the average word-vector of its fixed opening (its task definition), with a radius. The ball
around the signature is the task's protected zone. Word-vectors come from the frozen embedding table
of the base model, so nothing here is trained; in the average, a word counts less the more common it
is in unrelated text (weight a / (a + frequency), SILENT_SIF=a).
An input inside a protected zone belongs to the OLDEST task whose zone contains it, and is answered
with the adapters that existed when that task was learned. Zones are never changed and older tasks
come first, so the answer on such an input can never change again, whatever is learned later
(a "certified" answer). An input outside every zone goes to the nearest zone, measured in units of
that zone's radius (best effort, no guarantee). The same file can also re-test every input with reworded task
definitions and under the other rules (exact key, all adapters on), see _extra_eval.

SILENT_BASELINE=1 changes nothing in the official code: it runs an official method (InfLoRA, GainLoRA)
as it is, and only adds the same re-test with reworded task definitions after the official test.

Settings are read from environment variables so that the official argument parser is left alone.
"""
import json
import os
import random
import runpy
import sys

REPO = os.getcwd()
sys.path.insert(0, os.path.join(REPO, "src"))

import torch
import torch.nn.functional as F
from transformers import GenerationConfig

from t5_inflora import LoRALayer, T5Attention, T5Stack  # the official model code (used for inflora and olora)
from cl_trainer_olora import OLoRATrainer, skip_instructions   # the official plain-adapter trainer
from compute_metrics import metric_max_over_ground_truths, rougeL_score

LAM = float(os.environ.get("SILENT_LAMBDA", "1.0"))            # weight of the silence loss
N_POOL = int(os.environ.get("SILENT_POOL", "400"))             # stand-in inputs per task
NEG_BATCH = int(os.environ.get("SILENT_NEG_BATCH", "8"))       # stand-in inputs per training step
MAX_LEN = int(os.environ.get("SILENT_MAX_LEN", "256"))         # stand-in inputs are cut to this many tokens
KINDS = os.environ.get("SILENT_KINDS", "a,b").split(",")
LOCK = os.environ.get("SILENT_LOCK", "0") == "1"                # the exact switch
BASELINE = os.environ.get("SILENT_BASELINE", "0") == "1"        # an official method, untouched; only the re-test is added
POOL_TASKS = os.environ.get("SILENT_POOL_TASKS", "mnli,cb,qqp,rte,dbpedia,agnews,yahoo,multirc,boolq,wic").split(",")
TEMPLATE = "Definition: {d}\n\nNow complete the following example -\nInput: {x}\nOutput: "   # as in src/cl_dataset.py

# ---- a switch that turns the NEW adapter off (older adapters and the base model are untouched) ----
_official_forward = LoRALayer.forward
LoRALayer._silent_off = False


def _forward(self, x):
    out = _official_forward(self, x)
    return out * 0.0 if (LoRALayer._silent_off and getattr(self, "_is_new", False)) else out


if not BASELINE:
    LoRALayer.forward = _forward


def _source(definition, text):
    return TEMPLATE.replace("{d}", definition.strip()).replace("{x}", text)


def _build_pool(trainer, model):
    """Choose the stand-in inputs and record what the model answers on them before the new adapter learns anything."""
    for m in model.modules():
        if isinstance(m, T5Attention):
            m.lora_q._is_new = True
            m.lora_v._is_new = True
    rng = random.Random(1000 * int(trainer.args.seed) + int(getattr(trainer, "cur_task_id", 0)))
    defs, texts = [], []
    for t in POOL_TASKS:
        data = json.load(open(os.path.join(REPO, "CL_Benchmark", "Long_Sequence", t, "train.json")))
        d = data["Definition"][0] if isinstance(data["Definition"], list) and data["Definition"] else data["Definition"]
        assert isinstance(d, str) and d.strip(), f"unrelated task {t} has no definition"
        defs.append(d)
        texts.append([inst["input"] for inst in data["Instances"]])
    own = trainer.train_dataset
    sources = []
    for i in range(N_POOL):
        kind = KINDS[i % len(KINDS)]
        k = rng.randrange(len(defs))
        if kind == "a":
            sources.append(_source(defs[k], own[rng.randrange(len(own))]["Instance"]["sentence"]))
        elif kind == "c":                                              # a random piece of unrelated text as the definition
            words = rng.choice(texts[k]).split()
            n = rng.randint(8, 40)
            start = rng.randrange(max(1, len(words) - n))
            sources.append(_source(" ".join(words[start:start + n]) or defs[k], own[rng.randrange(len(own))]["Instance"]["sentence"]))
        else:
            sources.append(_source(defs[k], rng.choice(texts[k])))
    tok = trainer.tokenizer
    enc = tok(sources, max_length=MAX_LEN, truncation=True, padding=True, return_tensors="pt")
    device = next(model.parameters()).device
    gen_cfg = dict(max_new_tokens=50, num_beams=1, repetition_penalty=1.0, decoder_start_token_id=0,
                   eos_token_id=1, pad_token_id=0)                    # as in the official prediction step
    labels = torch.full((N_POOL, 50), -100, dtype=torch.long)
    was_training = model.training
    model.eval()
    LoRALayer._silent_off = True
    with torch.no_grad():
        for s in range(0, N_POOL, 16):
            ids, att = enc["input_ids"][s:s + 16].to(device), enc["attention_mask"][s:s + 16].to(device)
            out = model.generate(input_ids=ids, generation_config=GenerationConfig(attention_mask=att, **gen_cfg))
            out = out[:, 1:51].cpu()                                   # drop the start token
            out = out.masked_fill(out == 0, -100)                      # padding after the end
            labels[s:s + 16, :out.shape[1]] = out
    LoRALayer._silent_off = False
    model.train(was_training)
    empty = (labels != -100).sum(1) == 0
    labels[empty, 0] = 1                                               # an empty answer is just "end"
    order = list(range(N_POOL))
    rng.shuffle(order)
    print(f"[silent] {N_POOL} stand-in inputs ready (kinds {KINDS}); average answer length "
          f"{(labels != -100).sum(1).float().mean().item():.1f} tokens", flush=True)
    return {"ids": enc["input_ids"], "att": enc["attention_mask"], "labels": labels, "order": order, "pos": 0,
            "calls": 0, "sum": 0.0, "device": device}


def _silence_loss(trainer, model):
    st = getattr(trainer, "_silent_state", None)
    if st is None:
        st = trainer._silent_state = _build_pool(trainer, model)
    if st["pos"] + NEG_BATCH > len(st["order"]):
        random.Random(st["calls"]).shuffle(st["order"])
        st["pos"] = 0
    idx = st["order"][st["pos"]:st["pos"] + NEG_BATCH]
    st["pos"] += NEG_BATCH
    ids, att, labels = (st[k][idx].to(st["device"]) for k in ("ids", "att", "labels"))
    keep = att.sum(0) > 0                                              # drop columns that are padding for the whole batch
    ids, att = ids[:, keep], att[:, keep]
    model.eval()                                                       # no dropout in either pass, so they differ only by the adapter
    LoRALayer._silent_off = True
    with torch.no_grad():
        without = model(input_ids=ids, attention_mask=att, labels=labels).logits
    LoRALayer._silent_off = False
    with_new = model(input_ids=ids, attention_mask=att, labels=labels).logits
    model.train()
    mask = labels != -100
    kl = F.kl_div(F.log_softmax(with_new.float(), -1), F.softmax(without.float(), -1), reduction="none").sum(-1)
    loss = (kl * mask).sum() / mask.sum().clamp(min=1)
    st["calls"] += 1
    st["sum"] += loss.item()
    if st["calls"] == int(os.environ.get("SILENT_SELFCHECK", "0")):
        _selfcheck(model, ids, att, labels)
    if st["calls"] % 50 == 0:
        print(f"[silent] step {st['calls']}: average silence loss over the last 50 steps {st['sum'] / 50:.5f}", flush=True)
        st["sum"] = 0.0
    return loss


def _selfcheck(model, ids, att, labels):
    """After some training: 'new adapter switched off' must equal 'new adapter's weights set to zero'."""
    model.eval()
    with torch.no_grad():
        LoRALayer._silent_off = True
        off = model(input_ids=ids, attention_mask=att, labels=labels).logits
        LoRALayer._silent_off = False
        on = model(input_ids=ids, attention_mask=att, labels=labels).logits
        saved = [(m, m.lora_B.data.clone()) for m in model.modules() if isinstance(m, LoRALayer) and getattr(m, "_is_new", False)]
        for m, _ in saved:
            m.lora_B.data.zero_()
        zeroed = model(input_ids=ids, attention_mask=att, labels=labels).logits
        for m, b in saved:
            m.lora_B.data.copy_(b)
    model.train()
    print(f"[silent] self-check: switched-off vs zeroed weights differ by {(off - zeroed).abs().max().item():.2e} (must be ~0); "
          f"the trained adapter changes these outputs by {(on - off).abs().max().item():.2e}; new adapter layers: {len(saved)}", flush=True)
    assert (off - zeroed).abs().max().item() < 1e-4, "the off-switch does not match zeroed weights"


_official_compute_loss = OLoRATrainer.compute_loss


def _compute_loss(self, model, inputs, return_outputs=False):
    out = _official_compute_loss(self, model, inputs, return_outputs)
    if not model.training or LAM <= 0:
        return out
    extra = LAM * _silence_loss(self, model)
    return (out[0] + extra, out[1]) if return_outputs else out + extra


if not BASELINE:
    OLoRATrainer.compute_loss = _compute_loss


# ---- the exact switch (SILENT_LOCK=1) ----
_LOCKS = {"keys": {}, "t": None, "mask": None, "seen": 0, "unmatched": 0}


def _arg(name):
    return sys.argv[sys.argv.index(name) + 1] if name in sys.argv else None


def _lock_setup(trainer):
    """Store this task's key (the tokens of its definition) and load the keys of all earlier tasks."""
    tok, t = trainer.tokenizer, int(trainer.cur_task_id) + 1
    template = trainer.train_dataset[0]["Instance"]["instruction"]
    a = tok(template.replace("{0}", "A B"))["input_ids"]
    b = tok(template.replace("{0}", "zzq xkj"))["input_ids"]
    n = 0
    while n < min(len(a), len(b)) and a[n] == b[n]:
        n += 1
    key = torch.tensor(a[:n], dtype=torch.long)
    assert 5 <= len(key) < len(a), "could not find the task definition at the start of the input"
    keys = {}
    for d in (_arg("--previous_lora_path") or "").split(","):
        if d:
            k = torch.load(os.path.join(d, "task_key.pt"))
            keys[int(k["task_index"])] = k["key"]
    assert sorted(keys) == list(range(1, t)), f"keys of earlier tasks are missing: have {sorted(keys)}, need 1..{t - 1}"
    for k, old in keys.items():
        m = min(len(old), len(key))
        assert not torch.equal(old[:m], key[:m]), f"task {t} has the same definition as task {k}; the switch cannot tell them apart"
    keys[t] = key
    save_dir = os.path.join(trainer.args.output_dir, "saved_weights")
    os.makedirs(save_dir, exist_ok=True)
    torch.save({"task_index": t, "key": key}, os.path.join(save_dir, "task_key.pt"))
    _LOCKS.update(keys=keys, t=t)
    print(f"[locked] task {t}: key of {len(key)} tokens stored; {len(keys) - 1} earlier keys loaded", flush=True)


def _lock_mask(input_ids, count):
    """For every input: 1 for the adapters that existed when its task was learned, 0 for later ones."""
    t, B = _LOCKS["t"], input_ids.shape[0]
    matched = torch.zeros(B, dtype=torch.long, device=input_ids.device)
    best = torch.zeros(B, dtype=torch.long, device=input_ids.device)
    for k, key in _LOCKS["keys"].items():
        n = len(key)
        if input_ids.shape[1] >= n:
            hit = (input_ids[:, :n] == key.to(input_ids.device)).all(1) & (n > best)
            matched[hit], best[hit] = k, n
    if count:
        _LOCKS["seen"] += B
        _LOCKS["unmatched"] += int((matched == 0).sum())
    branch_task = torch.arange(t, 0, -1, device=input_ids.device)       # adapters are ordered newest first
    keep = (branch_task[None, :] <= matched[:, None]) | (matched[:, None] == 0)
    return keep.float().unsqueeze(-1)


_official_stack_forward = T5Stack.forward
_official_attention_forward = T5Attention.forward
_official_init = OLoRATrainer.__init__


def _stack_forward(self, *args, **kwargs):
    input_ids = kwargs.get("input_ids", args[0] if args else None)
    if not self.is_decoder and input_ids is not None and _LOCKS["t"]:
        _LOCKS["mask"] = _lock_mask(input_ids, count=not self.training)
    return _official_stack_forward(self, *args, **kwargs)


def _attention_forward(self, *args, **kwargs):
    w, mask = kwargs.get("key_attention_weights"), _LOCKS["mask"]
    if w is not None and mask is not None and w.shape[:2] == mask.shape[:2]:
        kwargs["key_attention_weights"] = w * mask.to(w.dtype)
    return _official_attention_forward(self, *args, **kwargs)


def _init(self, *args, **kwargs):
    _official_init(self, *args, **kwargs)
    _lock_setup(self)


if LOCK:
    T5Stack.forward = _stack_forward
    T5Attention.forward = _attention_forward
    OLoRATrainer.__init__ = _init


# ---- the certified switch (SILENT_CLAIM=1) ----
CLAIM = os.environ.get("SILENT_CLAIM", "0") == "1"
KAPPA = float(os.environ.get("SILENT_KAPPA", "0.5"))            # zone radius = KAPPA x distance to the nearest known signature
EXTRA_EVAL = os.environ.get("SILENT_EVAL", "1") == "1"          # re-test with reworded definitions and the other rules
BG_N = int(os.environ.get("SILENT_BG", "4000"))                # unrelated inputs used to set the scale of distances
SIF_A = float(os.environ.get("SILENT_SIF", "0.001"))            # common words count less: weight a / (a + frequency); 0 = all equal
HEAD = 64
RULES = ["certified", "nearest", "centre", "key", "all"]
LEVELS = ["original"]   # no-instruction run: the reworded definitions of the earlier run would put a task description back
# Reworded task definitions for the five-task test (written for this test; answer formats are unchanged).
REWORDED = {
    "task1510": [
        "Given a phrase that describes the relationship between two words, extract the two words and the lexical relation between them. The relation must be one of the types 'MemberOf', 'MadeOf', 'Synonym', 'Entails', 'HasA', 'HasProperty', 'PartOf', 'Antonym' or 'IsA'. The output should be in the format: word1 relation word2.",
        "You will see a short phrase that tells how two words are related. Find the two words and name the lexical relation that links them. Allowed relations are 'MemberOf', 'MadeOf', 'Synonym', 'Entails', 'HasA', 'HasProperty', 'PartOf', 'Antonym' and 'IsA'. Write your answer as: word1 relation word2.",
        "Read the phrase, pick out the pair of words it talks about, and say which link holds between them ('MemberOf', 'MadeOf', 'Synonym', 'Entails', 'HasA', 'HasProperty', 'PartOf', 'Antonym', 'IsA'). Answer format: word1 relation word2."],
    "task363": [
        'In this task, you are given a sentence from a movie review. Your task is to classify the sentence as "POS" if its sentiment is positive or as "NEG" if its sentiment is negative.',
        'You will read a sentence taken from a film review. Decide whether it expresses a positive or a negative opinion. Answer "POS" for positive and "NEG" for negative.',
        'Here is one line from a review of a movie. Label it "POS" when the writer liked what they saw and "NEG" when they did not.'],
    "task748": [
        'In this task, you are given a short story, and one sentence from the story is selected. Consider the events that happen after that sentence. Is any of them directly caused by it or made possible by it? Write your answer in the form " A >causes/enables> B". Try to use phrases and sentences from the story in your answer when possible. Do not change the selected sentence in your answer.',
        'You will read a short story in which one sentence has been picked out. Think about what happens later in the story and find an event that the picked sentence directly causes or makes possible. Give the answer as " A >causes/enables> B", reusing wording from the story where you can and keeping the picked sentence unchanged.',
        'A brief story follows, with one of its sentences marked. Name a later event that happens because of the marked sentence, or that it allows to happen. Use the format " A >causes/enables> B", copy text from the story when possible, and leave the marked sentence as it is.'],
    "task1687": [
        'In this task, you are given the text of a tweet. Your task is to classify the given tweet text into one of two categories, 1) positive or 2) negative, based on its content.',
        'You will read the text of a tweet. Decide from what it says whether the tweet is positive or negative, and answer with one of these two categories.',
        'Below is a post from Twitter. Say whether its mood is positive or negative.'],
    "task875": [
        "In this task, you are given a sentence that contains a particular emotion. You must classify the sentence into one of these emotions: 'joy', 'love', 'anger', 'fear', or 'surprise'.",
        "You will read a sentence that expresses a feeling. Choose which emotion it shows from this list: 'joy', 'love', 'anger', 'fear', 'surprise'.",
        "Which feeling does the writer of this sentence have? Pick one of 'joy', 'love', 'anger', 'fear', 'surprise'."],
}
_CL = {"t": None, "tasks": {}, "base": None, "emb": None, "weight": None, "pad": 0, "rule": "certified", "force": None,
       "seen": 0, "certified": 0, "trainer": None, "predict": None, "counts": None}


def _means(input_ids, lengths):
    """Average word-vector of the first n real tokens of every input, for every n in `lengths` (None: all tokens).
    Each token counts with its weight (all equal if no weights are set)."""
    emb, weight = _CL["emb"], _CL["weight"]
    input_ids = input_ids.to(emb.device)
    out = {n: [] for n in lengths}
    for s in range(0, input_ids.shape[0], 16):
        ids = input_ids[s:s + 16]
        w = (ids != _CL["pad"]).to(emb.dtype)                          # padding adds nothing
        if weight is not None:
            w = w * weight[ids]
        e = emb[ids] * w.unsqueeze(-1)
        for n in lengths:
            part, count = (e, w) if n is None else (e[:, :n], w[:, :n])
            out[n].append(part.sum(1) / count.sum(1).clamp(min=1e-12).unsqueeze(-1))
    return {n: torch.cat(v) for n, v in out.items()}


def _whitener(x):
    """Centre and rescale so that unrelated inputs vary by about the same amount in every direction."""
    x = x.cpu()
    mu = x.mean(0)
    cov = (x - mu).T @ (x - mu) / (x.shape[0] - 1)
    cov = 0.5 * cov + 0.5 * torch.eye(cov.shape[0], dtype=cov.dtype) * torch.trace(cov) / cov.shape[0]
    w, v = torch.linalg.eigh(cov)
    return mu, v @ torch.diag(w ** -0.5) @ v.T


def _prefix_len(rows):
    n = 0
    while all(len(r) > n for r in rows) and len({r[n] for r in rows}) == 1:
        n += 1
    return n


def _claim_base(tok, max_len):
    """The fixed scale of distances and the signatures of unrelated tasks. Made once, from unrelated data only."""
    rng = random.Random(0)
    defs, texts = [], []
    for name in POOL_TASKS:
        data = json.load(open(os.path.join(REPO, "CL_Benchmark", "Long_Sequence", name, "train.json")))
        d = data["Definition"][0] if isinstance(data["Definition"], list) and data["Definition"] else data["Definition"]
        assert isinstance(d, str) and d.strip(), f"unrelated task {name} has no definition"
        defs.append(d)
        texts.append([inst["input"] for inst in data["Instances"]])
    picks = [rng.randrange(len(defs)) for _ in range(BG_N)]
    sources = [_source(defs[k], rng.choice(texts[k])) for k in picks]
    ids = tok(sources, max_length=max_len, truncation=True, padding=True, return_tensors="pt")["input_ids"]
    base = {}
    if SIF_A > 0:                                                      # how common each token is in unrelated text
        counts = torch.bincount(ids[ids != tok.pad_token_id], minlength=_CL["emb"].shape[0]).double()
        base["weight"] = SIF_A / (SIF_A + counts / counts.sum())
        base["weight"][tok.pad_token_id] = 0.0
        _CL["weight"] = base["weight"].to(_CL["emb"].device)
    m = _means(ids, [HEAD, None])
    base["mu_head"], base["w_head"] = _whitener(m[HEAD])
    base["mu_all"], base["w_all"] = _whitener(m[None])
    sigs = []
    for d, xs in zip(defs, texts):
        rows = tok([_source(d, x) for x in xs[:20]], max_length=max_len, truncation=True)["input_ids"]
        n = _prefix_len(rows)
        assert n >= 5, "an unrelated task has no fixed opening"
        sigs.append((_means(torch.tensor([rows[0][:n]]), [n])[n][0].cpu() - base["mu_head"]) @ base["w_head"])
    base["bg_sig"] = torch.stack(sigs)
    base["emb_sum"] = float(_CL["emb"].sum())
    return base


def _routes(input_ids, rule):
    """For every input: the task it is given to (adapters of tasks 1..route are on), and whether that is certified."""
    t, tasks, base = _CL["t"], _CL["tasks"], _CL["base"]
    dev = _CL["emb"].device
    input_ids = input_ids.to(dev)
    B = input_ids.shape[0]
    route = torch.full((B,), t, dtype=torch.long, device=dev)
    cert = torch.zeros(B, dtype=torch.bool, device=dev)
    if rule == "all":                                                  # every adapter on (plain adapters)
        return route, cert
    if rule == "key":                                                  # the exact switch: the stored opening, token for token
        best = torch.zeros(B, dtype=torch.long, device=dev)
        for k in sorted(tasks):
            key = tasks[k]["key"]
            if input_ids.shape[1] >= len(key):
                hit = (input_ids[:, :len(key)] == key).all(1) & (len(key) > best)
                route[hit], best[hit] = k, len(key)
        return route, best > 0
    order = sorted(tasks)
    m = _means(input_ids, sorted({tasks[k]["n"] for k in order}) + [None])
    if rule == "centre":                                               # nearest centre of the whole input (an earlier fallback)
        whole = (m[None] - base["mu_all"]) @ base["w_all"]
        dist = torch.stack([(whole - tasks[k]["centre"]).norm(dim=1) for k in order], 1)
        return torch.tensor(order, device=dev)[dist.argmin(1)], cert
    # distance from every signature, in units of that zone's radius: 1 or less means inside the zone
    rel = torch.stack([(((m[tasks[k]["n"]] - base["mu_head"]) @ base["w_head"]) - tasks[k]["sig"]).norm(dim=1) / tasks[k]["rho"]
                       for k in order], 1)
    nearest = torch.tensor(order, device=dev)[rel.argmin(1)]
    if rule == "certified":
        for i, k in enumerate(order):                                  # oldest first: an older claim is never overruled
            new = (rel[:, i] <= 1.0) & ~cert
            route[new] = k
            cert |= new
    return torch.where(cert, route, nearest), cert


def _new_zone(key, train_batches):
    """The protected zone (signature and radius) and the centre of a new task, from its fixed opening and its
    training inputs. Uses only this task's inputs, the fixed scale, and the signatures already known."""
    base, n = _CL["base"], len(key)
    sig = (_means(key[None], [n])[n][0] - base["mu_head"]) @ base["w_head"]
    known = torch.cat([base["bg_sig"]] + [v["sig"][None] for v in _CL["tasks"].values()])
    nearest_known = float((known - sig).norm(dim=1).min())
    rows = [(_means(ids, [None])[None] - base["mu_all"]) @ base["w_all"] for ids in train_batches]
    return {"n": n, "key": key.to(sig.device), "sig": sig, "rho": KAPPA * nearest_known, "centre": torch.cat(rows).mean(0)}, nearest_known


def _claim_setup(trainer):
    """Make and store this task's protected zone and centre; load those of all earlier tasks."""
    tok, t, model = trainer.tokenizer, int(trainer.cur_task_id) + 1, trainer.model
    weight = model.encoder.embed_tokens.weight
    assert not weight.requires_grad, "the embedding table must stay frozen"
    _CL.update(emb=weight.detach().double(), pad=tok.pad_token_id, trainer=trainer)
    dev = weight.device
    max_len = int(_arg("--max_source_length") or 512)
    tasks, base = {}, None
    for d in (_arg("--previous_lora_path") or "").split(","):
        if d:
            f = torch.load(os.path.join(d, "task_claim.pt"), map_location="cpu")
            tasks[int(f["task_index"])], base = f["task"], f["base"]
    assert sorted(tasks) == list(range(1, t)), f"zones of earlier tasks are missing: have {sorted(tasks)}, need 1..{t - 1}"
    if base is None:
        base = _claim_base(tok, max_len)
    _CL["weight"] = base["weight"].to(dev) if base.get("weight") is not None else None
    assert abs(base["emb_sum"] - float(_CL["emb"].sum())) <= 1e-6 * abs(base["emb_sum"]), "the embedding table has changed"
    _CL["base"] = {k: (v.to(dev) if torch.is_tensor(v) else v) for k, v in base.items()}
    _CL["tasks"] = {k: {a: (b.to(dev) if torch.is_tensor(b) else b) for a, b in v.items()} for k, v in tasks.items()}
    _CL["t"] = t - 1

    template = trainer.train_dataset[0]["Instance"]["instruction"]
    a = tok(template.replace("{0}", "A B"))["input_ids"]
    b = tok(template.replace("{0}", "zzq xkj"))["input_ids"]
    n = _prefix_len([a, b])
    assert 5 <= n < len(a), "could not find the fixed opening of the input"
    data = trainer.train_dataset                                        # the task's own training inputs, as the model sees them
    batches = [trainer.data_collator([data[i] for i in range(s, min(s + 32, len(data)))])["input_ids"]
               for s in range(0, len(data), 32)]
    blocked = sum(int(_routes(ids, "certified")[1].sum()) for ids in batches) if t > 1 else 0   # already inside an older zone
    task, nearest_known = _new_zone(torch.tensor(a[:n], dtype=torch.long), batches)
    rho = task["rho"]
    _CL["tasks"][t] = task
    _CL["t"] = t
    save_dir = os.path.join(trainer.args.output_dir, "saved_weights")
    os.makedirs(save_dir, exist_ok=True)
    torch.save({"task_index": t, "task": {a: (b.cpu() if torch.is_tensor(b) else b) for a, b in task.items()}, "base": base},
               os.path.join(save_dir, "task_claim.pt"))
    print(f"[certified] task {t}: opening of {n} tokens; zone radius {rho:.2f} ({KAPPA} x {nearest_known:.2f}, the distance to the "
          f"nearest known signature); {len(tasks)} earlier zones loaded; training inputs already inside an older zone: "
          f"{blocked} of {len(trainer.train_dataset)}", flush=True)


def _claim_mask(input_ids, count):
    force = _CL["force"]
    if force is not None and force.shape[0] == input_ids.shape[0]:
        route = force.to(input_ids.device)
    else:
        route, cert = _routes(input_ids, _CL["rule"])
        route = route.to(input_ids.device)
        if count:
            _CL["seen"] += input_ids.shape[0]
            _CL["certified"] += int(cert.sum())
    branch_task = torch.arange(_CL["t"], 0, -1, device=input_ids.device)   # adapters are ordered newest first
    return (branch_task[None, :] <= route[:, None]).float().unsqueeze(-1)


def _stack_forward_claim(self, *args, **kwargs):
    input_ids = kwargs.get("input_ids", args[0] if args else None)
    if not self.is_decoder and input_ids is not None and _CL["t"]:
        _LOCKS["mask"] = _claim_mask(input_ids, count=not self.training)
    return _official_stack_forward(self, *args, **kwargs)


def _init_claim(self, *args, **kwargs):
    _official_init(self, *args, **kwargs)
    _claim_setup(self)


_official_predict = OLoRATrainer.predict


def _predict(self, test_dataset, *args, **kwargs):
    _CL["seen"] = _CL["certified"] = 0
    out = _official_predict(self, test_dataset, *args, **kwargs)
    _CL["predict"] = (test_dataset, out.predictions)
    _CL["counts"] = (_CL["seen"], _CL["certified"])
    return out


def _reworded(example, level):
    """The same test example with its task definition reworded (level 0: unchanged). None if there is no rewording."""
    if level == 0:
        return example
    new = REWORDED.get(example["Dataset"].split("_")[0])
    if new is None:
        return None
    inst = dict(example["Instance"])
    cut = inst["instruction"].rfind("Now complete the following example -")
    assert inst["instruction"].startswith("Definition: ") and cut > 0
    inst["instruction"] = "Definition: " + new[level - 1] + "\n\n" + inst["instruction"][cut:]
    return dict(example, Instance=inst)


def _extra_eval(rules):
    """Test every example again: with reworded definitions, and under each rule for choosing the adapters.
    An answer depends only on the input and on which adapters are on, so each (input, route) is generated once.
    With rules == ["official"] the model is used exactly as the official code left it (no route is forced)."""
    trainer, (dataset, official) = _CL["trainer"], _CL["predict"]
    model, tok, t = trainer.model, trainer.tokenizer, _CL["t"]
    device = next(model.parameters()).device
    model.eval()
    examples = [dataset[i] for i in range(len(dataset))]
    refs = {}
    for name in sorted({e["Dataset"] for e in examples}):
        data = json.load(open(os.path.join(_arg("--data_dir"), "SuperNI", name, "test.json")))
        refs[name] = [i["output"] if isinstance(i["output"], list) else [i["output"]] for i in data["Instances"]]
    gen = dict(max_new_tokens=50, num_beams=1, repetition_penalty=1.0, decoder_start_token_id=0, eos_token_id=1,
               pad_token_id=0, synced_gpus=False)                     # as in the official prediction step

    size = int(trainer.args.per_device_eval_batch_size)               # the batch size of the official test

    def batches(exs):
        for s in range(0, len(exs), size):
            b = trainer.data_collator(exs[s:s + size])
            yield b["input_ids"].to(device), b["attention_mask"].to(device)

    def generate(exs, route):
        preds = []
        for ids, att in batches(exs):
            _CL["force"] = None if route is None else torch.full((ids.shape[0],), route, dtype=torch.long, device=device)
            try:
                with torch.no_grad():
                    out = model.generate(input_ids=ids, generation_config=GenerationConfig(attention_mask=att, **gen))
            finally:
                _CL["force"] = None
            preds += skip_instructions(model, out.cpu().numpy(), tok)
        return preds

    result = {"task_index": t, "kappa": KAPPA, "sif_a": SIF_A, "rules": rules, "levels": {}}
    for li, level in enumerate(LEVELS):
        exs = [x for x in (_reworded(e, li) for e in examples) if x is not None]
        if not exs:
            continue
        routes = {rule: [] for rule in rules}
        certified = []
        for ids, _ in batches(exs):
            for rule in rules:
                if rule == "official":
                    routes[rule] += [None] * ids.shape[0]
                    continue
                r, c = _routes(ids, rule)
                routes[rule] += r.tolist()
                if rule == "certified":
                    certified += c.tolist()
        certified = certified or [False] * len(exs)
        preds = {}
        for r in sorted({x for rule in rules for x in routes[rule]}, key=lambda x: -1 if x is None else x):
            idx = [i for i in range(len(exs)) if any(routes[rule][i] == r for rule in rules)]
            for i, p in zip(idx, generate([exs[i] for i in idx], r)):
                preds[i, r] = p
        rows = []
        for i, e in enumerate(exs):
            ref = refs[e["Dataset"]][int(e["Instance"]["id"])]
            row = {"dataset": e["Dataset"], "id": e["Instance"]["id"], "in_zone": bool(certified[i])}
            for rule in rules:
                p = preds[i, routes[rule][i]]
                row[rule] = {"route": routes[rule][i], "pred": p,
                             "score": 100.0 * metric_max_over_ground_truths(rougeL_score, prediction=p, ground_truths=ref)}
            rows.append(row)
        result["levels"][level] = rows
        line = " | ".join(f"{rule} {sum(r[rule]['score'] for r in rows) / len(rows):.2f}" for rule in rules)
        print(f"[retest] after task {t}, {level} wording, {len(rows)} inputs: average score {line}; "
              f"inside a protected zone {sum(r['in_zone'] for r in rows)}", flush=True)
    if "original" in result["levels"]:                                 # this re-test must reproduce the official test
        mine = [r[rules[0]]["pred"] for r in result["levels"]["original"]]
        theirs = skip_instructions(model, official, tok)
        same = sum(a == b for a, b in zip(mine, theirs))
        result["same_as_official"] = [same, len(theirs)]
        print(f"[retest] check: this re-test gives the same answers as the official test on {same} of {len(theirs)} inputs", flush=True)
    json.dump(result, open(os.path.join(trainer.args.output_dir, "extra_eval.json"), "w"))


if CLAIM:
    assert not LOCK and not BASELINE, "use only one of SILENT_LOCK, SILENT_CLAIM, SILENT_BASELINE"
    T5Stack.forward = _stack_forward_claim
    T5Attention.forward = _attention_forward
    OLoRATrainer.__init__ = _init_claim
    OLoRATrainer.predict = _predict

if BASELINE:                                                           # nothing of the official method is changed:
    assert not LOCK, "use only one of SILENT_LOCK, SILENT_CLAIM, SILENT_BASELINE"
    from transformers import Seq2SeqTrainer                            # only the test set and its answers are noted
    _hf_predict = Seq2SeqTrainer.predict

    def _predict_any(self, test_dataset, *args, **kwargs):
        out = _hf_predict(self, test_dataset, *args, **kwargs)
        _CL.update(trainer=self, predict=(test_dataset, out.predictions), t=int(self.cur_task_id) + 1)
        return out

    Seq2SeqTrainer.predict = _predict_any

if __name__ == "__main__":
    if BASELINE:
        print("[baseline] official method, unchanged; a re-test with reworded definitions follows the official test", flush=True)
    else:
        print(f"[silent] silence training on: weight {LAM}, {N_POOL} stand-in inputs, {NEG_BATCH} per step, kinds {KINDS}",
              flush=True)
    if LOCK:
        print("[locked] exact switch on", flush=True)
    if CLAIM:
        print(f"[certified] certified switch on (radius factor {KAPPA}, word weights a = {SIF_A})", flush=True)
    script = os.path.join(REPO, "src", "run_t5.py")
    sys.argv = [script] + sys.argv[1:]
    try:
        runpy.run_path(script, run_name="__main__")
    finally:
        if LOCK:
            print(f"[locked] inputs seen while testing: {_LOCKS['seen']}; not matched to any task: {_LOCKS['unmatched']}", flush=True)
    if CLAIM and _CL["counts"]:
        print(f"[certified] official test: inputs routed {_CL['counts'][0]}; inside a protected zone {_CL['counts'][1]}", flush=True)
    if (CLAIM or BASELINE) and EXTRA_EVAL and _CL["predict"]:
        _extra_eval(RULES if CLAIM else ["official"])
'''
if SILENT_RUN_SOURCE is None:
    SILENT_RUN_SOURCE = open(os.path.join(os.path.dirname(os.path.abspath(__file__)), "silent_run.py")).read()
open(os.path.join(REPO, "silent_run.py"), "w").write(SILENT_RUN_SOURCE)
print("silent_run.py written:", len(SILENT_RUN_SOURCE.splitlines()), "lines")

## Remove the task definitions
The only change against the earlier run. In the downloaded copy of the official code, the "Definition" of
train, dev and test of every one of the five tasks becomes `[""]`. The official input builder then gives
`Definition: ` + empty text, and the same is true for the input builder of `silent_run.py`. Nothing else in the
data is touched (the examples, the answers and the unused fields stay). The cell checks every file afterwards.

In [ ]:
NOINST_TASKS = list(CFG["tasks"])
NOINST_EMPTY_INPUT = "Definition: \n\nNow complete the following example -\nInput: {0}\nOutput: "   # the input text, same for every task


def remove_definitions(repo, tasks):
    """Set "Definition" to [""] in train/dev/test of `tasks` everywhere under `repo` (outside the git folder and the
    run outputs). Returns {path: length of the definition that was removed}."""
    changed = {}
    for root, dirs, files in os.walk(repo):
        dirs[:] = [d for d in dirs if d not in (".git", "logs_and_outputs", "__pycache__")]
        if os.path.basename(root) in tasks:
            for f in files:
                if f in ("train.json", "dev.json", "test.json"):
                    p = os.path.join(root, f)
                    data = json.load(open(p, encoding="utf-8"))
                    if not isinstance(data, dict) or "Definition" not in data:
                        continue
                    old = data["Definition"]
                    changed[p] = len(old[0] if isinstance(old, list) and old else (old if isinstance(old, str) else ""))
                    data["Definition"] = [""]
                    tmp = p + ".tmp"
                    with open(tmp, "w", encoding="utf-8") as fh:
                        json.dump(data, fh, ensure_ascii=False)
                    os.replace(tmp, p)
    return changed


def check_definitions_removed(repo, tasks):
    """Every train/dev/test file of every task must now have Definition == [""] and still have its examples."""
    n = 0
    for t in tasks:
        for split in ("train", "dev", "test"):
            p = os.path.join(repo, "CL_Benchmark", "SuperNI", t, split + ".json")
            data = json.load(open(p, encoding="utf-8"))
            assert data["Definition"] == [""], f"definition not removed in {p}: {str(data['Definition'])[:80]}"
            assert len(data["Instances"]) > 0, f"no examples in {p}"
            n += 1
    return n


NOINST_CHANGED = remove_definitions(REPO, NOINST_TASKS)
assert len(NOINST_CHANGED) >= 3 * len(NOINST_TASKS), f"expected train, dev and test of every task; changed only {len(NOINST_CHANGED)} files"
print(f"definitions removed in {len(NOINST_CHANGED)} files; every one of the {check_definitions_removed(REPO, NOINST_TASKS)} "
      f"train/dev/test files of the {len(NOINST_TASKS)} tasks now has Definition == [\"\"]")
for p in NOINST_CHANGED:
    if "CL_Benchmark" + os.sep + "SuperNI" not in p:
        print("   also changed (another copy of a task):", p)
shutil.rmtree(os.path.join(CFG["work_dir"], "hf_cache", "datasets"), ignore_errors=True)   # no cached copy of the old data
json.dump({"task_definitions_removed": True, "tasks": NOINST_TASKS, "files": {os.path.relpath(p, REPO): n for p, n in NOINST_CHANGED.items()},
           "note": "values: length (characters) of the definition that was removed"}, open(os.path.join(SAVE_DIR, "noinstruction.json"), "w"), indent=1)

# The input that each code path builds for the first example of every task and split. Hard check: no task words.
code = ("import sys, json, types\nsys.path.insert(0, 'src')\nfrom cl_dataset import CLInstructions\n"
        "me = types.SimpleNamespace(_load_dataset=lambda p: CLInstructions._load_dataset(None, p))\n"
        "out = {}\nfor t, split in " + repr([(t, s) for t in NOINST_TASKS for s in ("train", "dev", "test")]) + ":\n"
        "    g = CLInstructions.load_SuperNI_dataset(me, 'CL_Benchmark/SuperNI/%s/%s.json' % (t, split), None, t, 'full', None, split)\n"
        "    out[t + '/' + split] = next(g)['Instance']['instruction']\nprint('OFFICIAL_INPUTS' + json.dumps(out))\n")
r = subprocess.run([ENV_PY, "-c", code], capture_output=True, text=True, env=RUN_ENV, cwd=REPO)
line = [l for l in r.stdout.splitlines() if l.startswith("OFFICIAL_INPUTS")]
if r.returncode == 0 and line:
    official_inputs = json.loads(line[0][len("OFFICIAL_INPUTS"):])
    assert all(v == NOINST_EMPTY_INPUT for v in official_inputs.values()), "an official input still contains something task-specific"
    print("official input builder: the text before the example is identical for all", len(official_inputs), "task/split pairs")
else:                                                  # only a check; the data themselves were checked above
    print("WARNING: could not run the official input builder as a check:\n" + (r.stdout + r.stderr)[-1500:])
first = json.load(open(os.path.join(REPO, "CL_Benchmark", "SuperNI", NOINST_TASKS[0], "test.json"), encoding="utf-8"))["Instances"][0]["input"]
print("input of the first test example of the first task, as the official code builds it:\n" + "-" * 30 + "\n"
      + NOINST_EMPTY_INPUT.replace("{0}", first) + "\n" + "-" * 30)

## The runs
Every setting that does not depend on the task order is read from the official scripts. The task
order, the folders and the per-task files are built here for our five tasks.

In [ ]:
def get(args, name):
    return args[args.index(name) + 1]


def put(args, name, value):
    if name in args:
        args[args.index(name) + 1] = str(value)
    else:
        args += [name, str(value)]


def copy_dir(src, dst):
    """Copy a folder so that `dst` only appears when the copy is complete."""
    tmp = dst + ".partial"
    shutil.rmtree(tmp, ignore_errors=True)
    shutil.copytree(src, tmp)
    shutil.rmtree(dst, ignore_errors=True)
    os.replace(tmp, dst)


def latest_bar(log_path):
    """The newest progress bar in the log, e.g. ' (latest progress bar: 310/500)'. Training, checking and
    final testing each have their own bar, so the numbers restart a few times within one task."""
    try:
        with open(log_path, "rb") as f:
            f.seek(0, 2); f.seek(max(0, f.tell() - 4000))
            bars = re.findall(r"(\d+)/(\d+) \[", f.read().decode(errors="replace"))
        return f" (latest progress bar: {bars[-1][0]}/{bars[-1][1]})" if bars else ""
    except OSError:
        return ""


ORDER = CFG["run_tag"]
for i, t in enumerate(CFG["tasks"]):                    # the small files that tell the official code which tasks to load
    d = os.path.join(REPO, "configs", f"{ORDER}_configs", t)
    os.makedirs(d, exist_ok=True)
    one = {"SuperNI": [{"sampling strategy": "full", "dataset name": t}]}
    json.dump(one, open(os.path.join(d, "train_tasks.json"), "w"), indent=2)
    json.dump(one, open(os.path.join(d, "dev_tasks.json"), "w"), indent=2)
    json.dump({"SuperNI": [{"sampling strategy": "full", "dataset name": x} for x in CFG["tasks"][:i + 1]]},
              open(os.path.join(d, "test_tasks.json"), "w"), indent=2)


def official_first_command(method):
    """The first command of the official script for this method: the source of all fixed settings."""
    text = open(os.path.join(REPO, f"gen_script_superni_order1_t5_{method}.sh")).read().replace("\\\n", " ")
    line = next(l for l in text.split("\n") if "python src/run_t5.py" in l)
    return shlex.split(line.split("python src/run_t5.py", 1)[1].replace("$2", "MODEL_PATH"))


def official_effective_batch():
    """Batch size times accumulation steps for each task, from the official script (32 for every task)."""
    text = open(os.path.join(REPO, "gen_script_superni_order1_t5_inflora.sh")).read().replace("\\\n", " ")
    out = {}
    for line in text.split("\n"):
        if "python src/run_t5.py" in line:
            a = shlex.split(line.split("python src/run_t5.py", 1)[1])
            out[os.path.basename(get(a, "--task_config_dir"))] = int(get(a, "--per_device_train_batch_size")) * \
                int(get(a, "--gradient_accumulation_steps"))
    return out


EFFECTIVE = official_effective_batch()


def build_command(method, i, run_id, batch, seed):
    tasks, t = CFG["tasks"], CFG["tasks"][i]
    out = f"logs_and_outputs/{run_id}/outputs"
    args = official_first_command("gainlora_inflora" if method == "gainlora_inflora" else "inflora")
    if method in OURS or method == "plain":             # the official plain-adapter mode, with its two penalties set to zero
        k = args.index("--threshold"); del args[k:k + 2]
        put(args, "--model_name", "olora"); put(args, "--lambda1", 0); put(args, "--lambda2", 0)
    put(args, "--model_name_or_path", CFG["model"])
    put(args, "--task_order", ",".join(tasks))
    put(args, "--task_config_dir", f"configs/{ORDER}_configs/{t}")
    put(args, "--output_dir", f"{out}/{i + 1}-{t}")
    put(args, "--metric_for_best_model", f"eval_rougeL_for_{t}")
    put(args, "--num_train_epochs", CFG["epochs"])
    put(args, "--per_device_train_batch_size", batch)
    put(args, "--gradient_accumulation_steps", max(1, EFFECTIVE[t] // batch))   # effective batch stays the same
    put(args, "--per_device_eval_batch_size", CFG["eval_batch"])
    put(args, "--run_name", run_id)
    put(args, "--seed", seed)
    if i > 0:                                           # the same extra arguments the official scripts give to later tasks
        prev = f"{out}/{i}-{tasks[i - 1]}/saved_weights"
        put(args, "--load_checkpoint_from", f"{prev}/trans_input.pt")
        put(args, "--previous_lora_path", ",".join(f"{out}/{j + 1}-{tasks[j]}/saved_weights" for j in range(i)))
        put(args, "--previous_prompt_key_path", f"{prev}/prompts_keys_till_now.pt")
        put(args, "--gen_data_dir", "generated_data/lora_gen_superni_t5")
        args.append("--add_instruction_replay")
    if TEST:                                            # tiny data for the CPU self-test
        put(args, "--max_train_samples", 96); put(args, "--max_eval_samples", 8)
        put(args, "--learning_rate", 0.01); put(args, "--num_train_epochs", 2)      # big changes, so forgetting shows up
    return args


OURS = ("silent", "silent2", "locked", "certified", "certified2")   # methods that go through silent_run.py
BASELINES = ("inflora", "gainlora_inflora")            # official methods; silent_run.py only adds the re-test to them


def run_method(method, seed, gpu=0, prefix=""):
    def say(text):
        print(prefix + text if prefix else text, flush=True)

    tasks = CFG["tasks"]
    run_id = f"{CFG['run_tag']}_{method}_seed{seed}"
    local_out = os.path.join(REPO, "logs_and_outputs", run_id, "outputs")
    saved_out = os.path.join(SAVE_DIR, run_id, "outputs")
    log_dir = os.path.join(SAVE_DIR, run_id, "logs")
    if (CFG.get(method, 0) is None or method in CFG["only_if_finished"]) and \
            not all(os.path.exists(os.path.join(saved_out, f"{i + 1}-{t}", "DONE")) for i, t in enumerate(tasks)):
        say(f"=== {method}: no finished copy in the results folder; it is only shown, never trained here (`only_if_finished`), skipped ===")
        return None
    os.makedirs(local_out, exist_ok=True); os.makedirs(saved_out, exist_ok=True); os.makedirs(log_dir, exist_ok=True)
    prog_path = os.path.join(SAVE_DIR, run_id, "progress.json")
    progress = json.load(open(prog_path)) if os.path.exists(prog_path) else {"tasks": {}}
    ours = method in OURS or (method in BASELINES and CFG["retest_baselines"])   # started through silent_run.py
    env = dict(RUN_ENV, **(CFG[method] or {})) if ours else dict(RUN_ENV)
    env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    script = "silent_run.py" if ours else "src/run_t5.py"
    say(f"=== {method}{', seed ' + str(seed) if len(CFG['seeds']) > 1 else ''}: {len(tasks)} tasks ===")
    for i, t in enumerate(tasks):
        name = f"{i + 1}-{t}"
        saved_task, local_task = os.path.join(saved_out, name), os.path.join(local_out, name)
        if os.path.exists(os.path.join(saved_task, "DONE")):
            if not os.path.exists(os.path.join(local_task, "DONE")):   # after a disconnect: bring it back from Drive
                shutil.rmtree(local_task, ignore_errors=True); shutil.copytree(saved_task, local_task)
            say(f"[{i + 1}/{len(tasks)}] {name}: already done, skipped")
            continue
        t0, log_path = time.time(), os.path.join(log_dir, name + ".log")
        say(f"[{i + 1}/{len(tasks)}] {name}: running ...")
        while True:
            batch = progress.get("train_batch", CFG["train_batch"])    # smaller if this method ran out of memory before
            args = build_command(method, i, run_id, batch, seed)
            shutil.rmtree(local_task, ignore_errors=True)              # remove anything left from an interrupted try
            with open(log_path, "w") as log:
                proc = subprocess.Popen([ENV_PY, script] + args, cwd=REPO, env=env, stdout=log, stderr=subprocess.STDOUT)
                while proc.poll() is None:
                    time.sleep(5 if TEST else 120)
                    if proc.poll() is None:
                        say(f"      ... {(time.time() - t0) / 60:.0f} min{latest_bar(log_path)}")
            if proc.returncode == 0 and os.path.exists(os.path.join(local_task, "all_results.json")):
                break
            log_text = open(log_path, errors="replace").read()
            if "out of memory" in log_text.lower() and batch > 2:      # GPU memory: halve the batch and try this task again
                progress["train_batch"] = batch // 2
                json.dump(progress, open(prog_path, "w"), indent=1)
                say(f"      the GPU ran out of memory with batch {batch}; trying again with batch {batch // 2} "
                    f"(the effective batch size does not change)")
                continue
            say(f"{name} failed. The end of its log:\n" + "".join(l[:300] + ("\n" if len(l) > 300 else "") for l in log_text.splitlines(True)[-40:]))
            raise RuntimeError(f"{name} failed; the full log is in {log_path}")
        for d in os.listdir(local_task):                               # as the official script does
            if d.startswith("checkpoint"):
                shutil.rmtree(os.path.join(local_task, d), ignore_errors=True)
        open(os.path.join(local_task, "DONE"), "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))
        copy_dir(local_task, saved_task)                               # now it is safe on Drive
        if i > 0 and not CFG["keep_all_files"]:                        # the previous task's helper matrices are no longer needed
            for base in (os.path.join(saved_out, f"{i}-{tasks[i - 1]}"), os.path.join(local_out, f"{i}-{tasks[i - 1]}")):
                for f in os.listdir(base):
                    if f.startswith("reg_") and f.endswith(".pt"):
                        os.remove(os.path.join(base, f))
                shutil.rmtree(os.path.join(base, "trans_input"), ignore_errors=True)
        log_text = open(log_path, errors="replace").read()
        silence = re.findall(r"average silence loss over the last 50 steps ([0-9.e+-]+)", log_text)
        lock = re.findall(r"inputs seen while testing: (\d+); not matched to any task: (\d+)", log_text)
        zone = re.findall(r"\[certified\] task \d+: .*", log_text)
        same = re.findall(r"same answers as the official test on (\d+) of (\d+)", log_text)
        progress["tasks"][name] = {"minutes": (time.time() - t0) / 60, "train_batch": batch,
                                   "silence_loss_first_last": [float(silence[0]), float(silence[-1])] if silence else None,
                                   "test_inputs_seen_unmatched": [int(lock[-1][0]), int(lock[-1][1])] if lock else None,
                                   "zone": zone[-1] if zone else None,
                                   "retest_same_as_official": [int(same[-1][0]), int(same[-1][1])] if same else None}
        json.dump(progress, open(prog_path, "w"), indent=1)
        say(f"      {name} done in {(time.time() - t0) / 60:.1f} min, saved")
        try:
            make_small_zip()
        except Exception as err:                         # never stops a run
            say(f"      (the small-files zip could not be refreshed: {err})")
    return run_id



import zipfile, threading
ZIP_LOCK = threading.Lock()


def make_small_zip():
    """The small result files (scores, every answer of the re-test, logs) in one archive; written after every finished task."""
    zip_path = os.path.join(CFG["drive_dir"], CFG["run_tag"] + "_small_files.zip")
    with ZIP_LOCK:
        tmp = zip_path + ".tmp"
        with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
            for root, _, files in os.walk(SAVE_DIR):
                if ".partial" in root:
                    continue
                for f in files:
                    if f.endswith((".json", ".jsonl", ".log", ".txt")) and not f.startswith(("tokenizer", "special_tokens", "spiece")):
                        try:
                            z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), CFG["drive_dir"]))
                        except OSError:
                            pass
        os.replace(tmp, zip_path)
    return zip_path


FAILED = {}                                            # runs that stopped with an error; the others carry on


def run_all(jobs):
    """Run every (method, seed): one after another, or two at a time when there are two GPUs (one run per GPU)."""
    many = len(CFG["seeds"]) > 1
    gpus = N_GPUS if CFG["use_all_gpus"] else 1
    done = {}

    def one(job, gpu, prefix):
        try:
            done[job] = run_method(job[0], job[1], gpu=gpu, prefix=prefix)
        except Exception as e:                                           # reported at the end; nothing else is stopped
            FAILED[f"{job[0]}, seed {job[1]}"] = str(e)
            print(f"{prefix}FAILED: {e}", flush=True)

    if gpus < 2 or len(jobs) < 2:
        for job in jobs:
            print()
            one(job, 0, "")
    else:
        import queue, threading
        print(f"{gpus} GPUs: {gpus} runs are trained at the same time, one per GPU. Lines are marked with the GPU and the run.")
        cost = {"gainlora_inflora": 3, "inflora": 2}                   # the slowest methods are started first
        todo = queue.Queue()
        for job in sorted(jobs, key=lambda job: -cost.get(job[0], 1)):
            todo.put(job)

        def worker(gpu):
            while True:
                try:
                    job = todo.get_nowait()
                except queue.Empty:
                    return
                one(job, gpu, f"[GPU {gpu}, {job[0]}{', seed ' + str(job[1]) if many else ''}] ")

        threads = [threading.Thread(target=worker, args=(g,)) for g in range(gpus)]
        for t in threads:
            t.start()
        for t in threads:
            t.join()
    return {job: run_id for job, run_id in done.items() if run_id}


RUNS = run_all([(m, s) for s in CFG["seeds"] for m in CFG["methods"]])

## Scores
**AP**: average score over the tasks at the very end (higher is better). **FT**: forgetting, how far
earlier tasks fell from their best score (lower is better). **learned**: average score of each task
right after it was learned. Scores are Rouge-L on about 100 test examples per task, so differences of
a few points on one task are noise.

In [ ]:
def scores_of(run_id):
    out = os.path.join(SAVE_DIR, run_id, "outputs")
    rows = []
    for i, t in enumerate(CFG["tasks"]):
        f = os.path.join(out, f"{i + 1}-{t}", "all_results.json")
        if not os.path.exists(os.path.join(out, f"{i + 1}-{t}", "DONE")):
            break
        res = json.load(open(f))
        rows.append([res[f"predict_eval_rougeL_for_{CFG['tasks'][j]}"] for j in range(i + 1)])
    return rows


def retest_table(run_id):
    """The re-test stored by a run: every rule and every wording, from the same trained weights."""
    out, tasks = os.path.join(SAVE_DIR, run_id, "outputs"), CFG["tasks"]
    steps = []
    for i, t in enumerate(tasks):
        f = os.path.join(out, f"{i + 1}-{t}", "extra_eval.json")
        if not (os.path.exists(f) and os.path.exists(os.path.join(out, f"{i + 1}-{t}", "DONE"))):
            break
        steps.append(json.load(open(f)))
    if not steps:
        return None
    T, table = len(steps), {}
    rules = steps[-1].get("rules") or ["certified", "nearest", "key", "all"]
    for level, last in steps[-1]["levels"].items():
        have = [j for j in range(T) if any(r["dataset"] == tasks[j] for r in last)]   # tasks that have this wording
        for rule in rules:
            score, by_id = {}, []
            for i in range(T):                                   # after task i+1: score of every task learned so far
                data = steps[i]["levels"].get(level, [])
                by_id.append({(r["dataset"], r["id"]): r for r in data})
                for j in have:
                    if j <= i:
                        s = [r[rule]["score"] for r in data if r["dataset"] == tasks[j]]
                        score[i, j] = sum(s) / len(s)
            earlier = [j for j in have if j < T - 1]
            changed = total = 0
            for j in earlier:                                    # answers right after learning task j+1 against the end
                for key, r in by_id[j].items():
                    if key[0] == tasks[j]:
                        total += 1
                        changed += r[rule]["pred"] != by_id[-1][key][rule]["pred"]
            routed = [r for r in last if r[rule]["route"] is not None]
            table[level, rule] = {
                "tasks": len(have), "learned": sum(score[j, j] for j in have) / len(have),
                "AP": sum(score[T - 1, j] for j in have) / len(have),
                "FT": sum(max(score[i, j] for i in range(j, T - 1)) - score[T - 1, j] for j in earlier) / len(earlier) if earlier else 0.0,
                "changed": changed, "compared": total,
                "right_task": sum(r[rule]["route"] == tasks.index(r["dataset"]) + 1 for r in routed) / len(routed) if routed else None,
                "certified": sum(r["in_zone"] for r in last), "inputs": len(last),
                "final_scores": {tasks[j]: score[T - 1, j] for j in have}, "learned_scores": {tasks[j]: score[j, j] for j in have}}
    return {"tasks_done": T, "table": {f"{a}|{b}": v for (a, b), v in table.items()},
            "same_as_official": [s.get("same_as_official") for s in steps]}


NAMES = {"certified": "certified (ours)", "nearest": "nearest only", "centre": "nearest centre", "key": "exact key (= locked)",
         "all": "all on (= plain)", "official": "as published"}


def report(run_ids):
    """Print the scores of the runs of one seed and return them as numbers."""
    summary = {}
    for method, run_id in run_ids.items():
        rows = scores_of(run_id)
        if not rows:
            continue
        T = len(rows)
        prog = json.load(open(os.path.join(SAVE_DIR, run_id, "progress.json")))
        summary[method] = {
            "tasks_done": T, "AP": sum(rows[-1]) / T,
            "FT": sum(max(r[j] for r in rows[j:-1]) - rows[-1][j] for j in range(T - 1)) / (T - 1) if T > 1 else 0.0,
            "learned": sum(rows[i][i] for i in range(T)) / T,
            "minutes": sum(x["minutes"] for x in prog["tasks"].values()), "rows": rows,
            "silence_loss_first_last": [x.get("silence_loss_first_last") for x in prog["tasks"].values()],
            "unmatched": [x.get("test_inputs_seen_unmatched") for x in prog["tasks"].values()],
        }

    print(f"{'method':<20}{'tasks':>6}{'learned':>10}{'AP':>8}{'FT':>8}{'minutes':>10}")
    for m, s in summary.items():
        print(f"{m:<20}{s['tasks_done']:>6}{s['learned']:>10.2f}{s['AP']:>8.2f}{s['FT']:>8.2f}{s['minutes']:>10.0f}")
    print("\neach task: score right after it was learned -> score at the end")
    print(f"{'task':<40}" + "".join(f"{m:>22}" for m in summary))
    for j, t in enumerate(CFG["tasks"]):
        cells = [f"{s['rows'][j][j]:6.1f} -> {s['rows'][-1][j]:5.1f}" if len(s["rows"]) > j else "-" for s in summary.values()]
        print(f"{t[:38]:<40}" + "".join(f"{c:>22}" for c in cells))
    for m in ("silent", "silent2"):
        if m in summary:
            print(f"\n{m}: silence loss per task (first 50 steps, last 50 steps):", summary[m]["silence_loss_first_last"])
    if "locked" in summary:
        print("\nlocked: test inputs (seen, not matched to a task) after each task:", summary["locked"]["unmatched"])
        same = all(abs(summary["locked"]["rows"][j][j] - summary["locked"]["rows"][-1][j]) < 1e-9 for j in range(len(summary["locked"]["rows"])))
        print("locked: every task scores exactly the same at the end as right after it was learned:", same)
    all_retests = {}
    for method, run_id in run_ids.items():
        retest = retest_table(run_id)
        if not retest:
            continue
        all_retests[method] = retest
        if method.startswith("certified"):
            prog = json.load(open(os.path.join(SAVE_DIR, run_id, "progress.json")))
            print(f"\n{method}: the protected zone of each task")
            for name, x in prog["tasks"].items():
                print("  ", x.get("zone"))
        print(f"\n{method} run, re-test after {retest['tasks_done']} tasks. Scores here use every reference answer (the official score")
        print("picks one at random), so they can differ a little from the table above. `tasks`: how many tasks have this wording.")
        print(f"{'wording':<10}{'rule':<22}{'tasks':>6}{'learned':>9}{'AP':>8}{'FT':>8}{'answers changed':>18}{'right task':>12}{'certified':>14}")
        for key, v in retest["table"].items():
            level, rule = key.split("|")
            right = f"{v['right_task']:.3f}" if (v["right_task"] is not None and rule != "all") else ""
            cert = f"{v['certified']} of {v['inputs']}" if rule == "certified" else ""
            print(f"{level:<10}{NAMES.get(rule, rule):<22}{v['tasks']:>6}{v['learned']:>9.2f}{v['AP']:>8.2f}{v['FT']:>8.2f}"
                  f"{str(v['changed']) + ' of ' + str(v['compared']):>18}{right:>12}{cert:>14}")
        print("re-test equals the official test (same answers, of inputs), per task:", retest["same_as_official"])
    if all_retests:
        print("\nanswers changed: answers to a task's test inputs that differ between right after learning it and the end.")
        print("right task: share of test inputs at the end that were given to their own task. certified: inputs inside a protected zone.")
    return {"summary": summary, "retest": all_retests}


def spread(values):
    return f"{sum(values) / len(values):.2f} ({min(values):.2f} to {max(values):.2f})"


BY_SEED = {}
for seed in CFG["seeds"]:
    if len(CFG["seeds"]) > 1:
        print("\n" + "=" * 40 + f" seed {seed} " + "=" * 40)
    BY_SEED[str(seed)] = report({m: RUNS[m, seed] for m in CFG["methods"] if (m, seed) in RUNS})

if len(CFG["seeds"]) > 1:
    full = len(CFG["tasks"])
    print("\n" + "=" * 40 + " all seeds together " + "=" * 40)
    print("Mean over the seeds, with the lowest and the highest seed in brackets. Only seeds in which the method finished every task.")
    print(f"{'method':<20}{'seeds':>6}{'learned':>26}{'AP':>26}{'FT':>26}")
    for m in CFG["methods"]:
        done = [b["summary"][m] for b in BY_SEED.values() if m in b["summary"] and b["summary"][m]["tasks_done"] == full]
        if done:
            print(f"{m:<20}{len(done):>6}" + "".join(f"{spread([d[k] for d in done]):>26}" for k in ("learned", "AP", "FT")))
    print("\nre-test (every reference answer): mean over the seeds; answers changed are added up")
    print(f"{'method':<20}{'wording':<10}{'rule':<22}{'seeds':>6}{'learned':>9}{'AP':>8}{'FT':>8}{'answers changed':>20}")
    for m in CFG["methods"]:
        tables = [b["retest"][m]["table"] for b in BY_SEED.values() if m in b["retest"] and b["retest"][m]["tasks_done"] == full]
        for key in (tables[0] if tables else []):
            level, rule = key.split("|")
            vals = [t[key] for t in tables if key in t]
            print(f"{m:<20}{level:<10}{NAMES.get(rule, rule):<22}{len(vals):>6}"
                  + "".join(f"{sum(v[k] for v in vals) / len(vals):>{w}.2f}" for k, w in (("learned", 9), ("AP", 8), ("FT", 8)))
                  + f"{str(sum(v['changed'] for v in vals)) + ' of ' + str(sum(v['compared'] for v in vals)):>20}")

# ---- the 100-epoch run against the 10-epoch runs of the same seed (42) ----
# Sources: no definitions, 10 epochs: silent_adapters/results/2026-11-18_kaggle_noinstruction_published_five_tasks/small_files.zip;
# with definitions, 10 epochs: silent_adapters/results/2026-11-17_kaggle_five_tasks_seed42/small_files.zip (both read with this
# notebook's own functions scores_of and retest_table).
PUBLISHED_10_EPOCHS = {'inflora': {'official': {'tasks_done': 5, 'learned': 51.08274, 'AP': 37.278980000000004, 'FT': 17.7547, 'minutes_per_task': [2.0, 4.0, 6.0, 4.0, 4.0]}, 'retest': {'official': {'tasks': 5, 'learned': 51.46030605643466, 'AP': 37.452831660868405, 'FT': 18.009342994457818, 'changed': 217, 'compared': 400}}}, 'gainlora_inflora': {'official': {'tasks_done': 5, 'learned': 55.81694, 'AP': 51.63982, 'FT': 6.5547499999999985, 'minutes_per_task': [4.0, 4.0, 6.0, 4.0, 4.0]}, 'retest': {'official': {'tasks': 5, 'learned': 56.08429545816297, 'AP': 51.94169771038794, 'FT': 6.511580518052115, 'changed': 129, 'compared': 400}}}}
EARLIER_WITH_DEFINITIONS = {'certified2': {'official': {'tasks_done': 5, 'learned': 78.69865999999999, 'AP': 78.87912, 'FT': 0.0, 'minutes_per_task': [6.0, 6.0, 10.0, 6.0, 8.0]}, 'retest': {'certified': {'tasks': 5, 'learned': 79.32603975124638, 'AP': 79.32603975124638, 'FT': 0.0, 'changed': 0, 'compared': 400}, 'nearest': {'tasks': 5, 'learned': 79.32603975124638, 'AP': 79.32603975124638, 'FT': 0.0, 'changed': 0, 'compared': 400}, 'centre': {'tasks': 5, 'learned': 79.32603975124638, 'AP': 79.32603975124638, 'FT': 0.0, 'changed': 0, 'compared': 400}, 'key': {'tasks': 5, 'learned': 79.32603975124638, 'AP': 79.32603975124638, 'FT': 0.0, 'changed': 0, 'compared': 400}, 'all': {'tasks': 5, 'learned': 79.32603975124638, 'AP': 33.5435830182889, 'FT': 57.22807091619686, 'changed': 331, 'compared': 400}}}, 'inflora': {'official': {'tasks_done': 5, 'learned': 56.689519999999995, 'AP': 58.372800000000005, 'FT': -1.1041000000000025, 'minutes_per_task': [6.0, 6.0, 10.0, 6.0, 8.0]}, 'retest': {'official': {'tasks': 5, 'learned': 56.96046595529524, 'AP': 58.88896381310951, 'FT': -1.410622322267848, 'changed': 98, 'compared': 400}}}, 'gainlora_inflora': {'official': {'tasks_done': 5, 'learned': 60.30426, 'AP': 61.0536, 'FT': -0.686675000000001, 'minutes_per_task': [6.0, 6.0, 10.0, 6.0, 8.0]}, 'retest': {'official': {'tasks': 5, 'learned': 60.851867946540246, 'AP': 61.311313690834666, 'FT': -0.3243071803680184, 'changed': 17, 'compared': 400}}}}
EARLIER_SOURCE = "10 epochs: silent_adapters/results/2026-11-18_kaggle_noinstruction_published_five_tasks (no definitions) and 2026-11-17_kaggle_five_tasks_seed42 (with definitions)"


def compare_with_earlier():
    here = BY_SEED.get("42", {"summary": {}, "retest": {}})
    print("\n" + "=" * 20 + " 100 epochs against 10 epochs, no task definition, seed 42 " + "=" * 20)
    print("learned: average score right after learning each task | AP: average score at the end | FT: forgetting (lower is better)")
    print("answers changed: answers to the inputs of the first four tasks (400) that differ between right after learning and the end")
    result = {}
    print(f"\n{'method':<20}{'run':<34}{'learned':>9}{'AP':>8}{'FT':>8}{'answers changed':>18}")
    for m in CFG["methods"]:
        now = (here["retest"].get(m) or {}).get("table", {}).get("original|official")
        old = PUBLISHED_10_EPOCHS.get(m, {}).get("retest", {}).get("official")
        wd = EARLIER_WITH_DEFINITIONS.get(m, {}).get("retest", {}).get("official")
        result[m] = {"retest_100_epochs": now and {k: now[k] for k in ("learned", "AP", "FT", "changed", "compared")}, "retest_10_epochs_no_definition": old,
                     "retest_10_epochs_with_definition": wd}
        for label, v in (("100 epochs, no definitions", now), ("10 epochs, no definitions", old), ("10 epochs, with definitions", wd)):
            if v is None:
                print(f"{m if label.startswith('100') else '':<20}{label:<34}{'(not available)':>43}")
            else:
                print(f"{m if label.startswith('100') else '':<20}{label:<34}{v['learned']:>9.2f}{v['AP']:>8.2f}{v['FT']:>8.2f}{str(v['changed']) + ' of ' + str(v['compared']):>18}")
    print("\nofficial test (one random reference answer per example):")
    print(f"{'method':<20}{'run':<34}{'learned':>9}{'AP':>8}{'FT':>8}")
    for m in CFG["methods"]:
        now, old = here["summary"].get(m), PUBLISHED_10_EPOCHS.get(m, {}).get("official")
        result[m + "|official_test"] = {"100_epochs": now and {k: now[k] for k in ("learned", "AP", "FT", "tasks_done")}, "10_epochs_no_definition": old}
        for label, v in (("100 epochs, no definitions", now), ("10 epochs, no definitions", old)):
            if v is None:
                print(f"{m if label.startswith('100') else '':<20}{label:<34}{'(not available)':>25}")
            else:
                print(f"{m if label.startswith('100') else '':<20}{label:<34}{v['learned']:>9.2f}{v['AP']:>8.2f}{v['FT']:>8.2f}")
    return result


COMPARISON = compare_with_earlier()

json.dump({"cfg": CFG, "svd_backend": SVD_BACKEND, "by_seed": BY_SEED, "failed": FAILED, "no_task_definition": True,
           "comparison_with_definitions": COMPARISON, "earlier_source": EARLIER_SOURCE},
          open(os.path.join(SAVE_DIR, "results.json"), "w"), indent=1)

ZIP = make_small_zip()
print("\nsaved:", os.path.join(SAVE_DIR, "results.json"), "| scores, answers and logs in one file:", ZIP)
if FAILED:
    print("\nRUNS THAT FAILED (they are missing or incomplete in the tables above; run all again to continue them):")
    for name, why in FAILED.items():
        print("  ", name, "-", why[:300])
print(f"{len(CFG['seeds'])} seed(s), {len(CFG['tasks'])} tasks, reduced setting: an early look, not a result. Copy everything above into the chat.")